#  **Mapa de Criminalidad Municipal, Colombia.**

Análisis geoespacial de tasas de criminalidad por municipio usando datos abiertos de la Policía Nacional (SIEDCO) y proyecciones de población del DANE.

**Delitos analizados:** Homicidios, Hurto a personas, Delitos sexuales

**Fuentes:**
- [Datos Abiertos Colombia](https://datos.gov.co) — Policía Nacional / DIJIN
- [DANE](https://www.dane.gov.co) — Proyecciones de población municipal 2020-2035
- [GeoJSON Municipal](https://github.com/caticoa3/colombia_mapa) — Geometrías DANE

**Métrica principal:** Tasa por cada 100,000 habitantes = (delitos / población) × 100,000

In [1]:
#SETUP
!pip install sodapy geopandas -q

import pandas as pd
import numpy as np
import geopandas as gpd
import plotly.express as px
from sodapy import Socrata

## ***1. Ingesta de datos***

In [2]:
#Descarga de delitos
client = Socrata("www.datos.gov.co", None)

def descargar_dataset(dataset_id, nombre):
    """Descarga un dataset completo paginando la API SODA."""
    all_results = []
    offset = 0
    while True:
        r = client.get(dataset_id, limit=50000, offset=offset)
        if not r:
            break
        all_results.extend(r)
        offset += 50000
    print(f"{nombre}: {len(all_results):,} registros")
    return pd.DataFrame.from_records(all_results)

df_homicidios = descargar_dataset("m8fd-ahd9", "Homicidios")
df_hurto = descargar_dataset("4rxi-8m8d", "Hurto a personas")
df_delitos_sex = descargar_dataset("fpe5-yrmw", "Delitos sexuales")

Homicidios: 335,300 registros
Hurto a personas: 638,569 registros
Delitos sexuales: 392,576 registros


In [3]:
#Descarga de población y geometrías
# Población municipal — DANE
pop_url = "https://www.dane.gov.co/files/censo2018/proyecciones-de-poblacion/Municipal/DCD-area-proypoblacion-Mun-2020-2035-ActPostCOVID-19.xlsx"
df_pop_raw = pd.read_excel(pop_url, header=11)
df_pop_raw.columns = ["cod_depto", "departamento", "cod_muni", "municipio", "anio", "area", "poblacion"]

# Recuperar fila perdida por header
fila_perdida = pd.DataFrame([{
    "cod_depto": "05", "departamento": "Antioquia", "cod_muni": "05001",
    "municipio": "Medellín", "anio": 2020, "area": "Total", "poblacion": 2519592
}])
df_pop_raw = pd.concat([fila_perdida, df_pop_raw], ignore_index=True)

df_pop = df_pop_raw[df_pop_raw["area"] == "Total"].copy()
df_pop["anio"] = df_pop["anio"].astype(int)
df_pop["poblacion"] = df_pop["poblacion"].astype(int)
df_pop["cod_muni"] = df_pop["cod_muni"].astype(str).str.split(".").str[0].str.zfill(5)

print(f"Población: {df_pop['cod_muni'].nunique()} municipios, años {df_pop['anio'].min()}-{df_pop['anio'].max()}")

# GeoJSON municipal
gdf_muni = gpd.read_file("https://raw.githubusercontent.com/caticoa3/colombia_mapa/master/co_2018_MGN_MPIO_POLITICO.geojson")
print(f"GeoJSON: {len(gdf_muni)} municipios")

Población: 1122 municipios, años 2020-2035
GeoJSON: 1122 municipios


## ***2. Limpieza y transformación***

In [6]:
def limpiar_dataset(raw, tipo_delito, col_codigo="cod_muni"):
    """Limpia un dataset de delitos: tipado, normalización de código municipal y filtro temporal."""
    d = raw.copy()
    d["fecha_hecho"] = pd.to_datetime(d["fecha_hecho"], errors="coerce")
    d["cantidad"] = pd.to_numeric(d["cantidad"], errors="coerce").fillna(0).astype(int)

    d[col_codigo] = d[col_codigo].astype(str).str.replace(".0", "", regex=False)
    if d[col_codigo].str.len().max() > 5:
        d[col_codigo] = d[col_codigo].str[:5]
    d[col_codigo] = d[col_codigo].str.zfill(5)

    d["anio"] = d["fecha_hecho"].dt.year
    d["tipo_delito"] = tipo_delito
    d = d[d["anio"] >= 2020]

    return d[[col_codigo, "anio", "cantidad", "tipo_delito"]].rename(columns={col_codigo: "cod_muni"})

hom = limpiar_dataset(df_homicidios, "Homicidio")
hur = limpiar_dataset(df_hurto, "Hurto a personas")
sex = limpiar_dataset(df_delitos_sex, "Delitos sexuales", col_codigo="codigo_dane")

df_todos = pd.concat([hom, hur, sex], ignore_index=True)

print(f"Total registros desde 2020: {len(df_todos):,}")
print(f"\nCasos por delito:")
print(df_todos.groupby("tipo_delito")["cantidad"].sum().to_frame("total"))

Total registros desde 2020: 424,284

Casos por delito:
                    total
tipo_delito              
Delitos sexuales    80874
Homicidio           84866
Hurto a personas  1929310


## ***3. Cálculo de tasas per cápita***

$$\text{Tasa} = \frac{\text{Número de delitos}}{\text{Población municipal}} \times 100{,}000$$

Esto permite hacer comparaciones más justas entre municipios con poblaciones diferentes. Analizar únicamente el número absoluto de homicidios puede ser engañoso, pues los municipios más grandes (habitantes) tienden a registrar más casos simplemente por tener más habitantes.

Por ejemplo, un municipio con 50 homicidios y 1 millón de habitantes tiene una tasa de 5 homicidios por cada 100.000 habitantes, mientras que otro con 10 homicidios y 20.000 habitantes alcanza una tasa de 50 por cada 100.000 habitantes. Aunque el segundo municipio presenta menos homicidios en términos absolutos, proporcionalmente enfrenta un nivel de violencia mucho mayor.

In [13]:
#tasas
agrupado = (
    df_todos.groupby(["anio", "cod_muni", "tipo_delito"])["cantidad"]
    .sum()
    .reset_index()
)

agrupado = agrupado.merge(
    df_pop[["cod_muni", "anio", "poblacion"]],
    on=["cod_muni", "anio"],
    how="left"
)

agrupado["tasa"] = (agrupado["cantidad"] / agrupado["poblacion"]) * 100_000
agrupado = agrupado.dropna(subset=["tasa"])

ultimo_anio = int(sorted(agrupado["anio"].unique())[-2])

print(f"Año de análisis: {ultimo_anio}")
print(f"\nTasa promedio por delito (por cada 100,000 hab.):")
print(
    agrupado[agrupado["anio"] == ultimo_anio]
    .groupby("tipo_delito")["tasa"]
    .mean()
    .round(1)
    .to_frame("tasa_promedio")
)

Año de análisis: 2025

Tasa promedio por delito (por cada 100,000 hab.):
                  tasa_promedio
tipo_delito                    
Delitos sexuales           26.8
Homicidio                  35.6
Hurto a personas          123.0


## ***4. Ranking municipal***

In [14]:
for delito in ["Homicidio", "Hurto a personas", "Delitos sexuales"]:
    datos = agrupado[(agrupado["anio"] == ultimo_anio) & (agrupado["tipo_delito"] == delito)]
    datos = datos.merge(
        df_pop[["cod_muni", "municipio"]].drop_duplicates(),
        on="cod_muni",
        how="left"
    )
    top = datos.nlargest(15, "tasa")[["municipio", "cod_muni", "cantidad", "poblacion", "tasa"]]
    print(f"\n{'='*60}")
    print(f"TOP 15 — {delito} (tasa x 100K hab.) — {ultimo_anio}")
    print(f"{'='*60}")
    print(top.to_string(index=False))


TOP 15 — Homicidio (tasa x 100K hab.) — 2025
             municipio cod_muni  cantidad  poblacion       tasa
San Andrés de Cuerquía    05647        29     7697.0 376.770170
               Calamar    95015        34    11935.0 284.876414
                  Tibú    54810       157    62474.0 251.304543
              Guachené    19300        50    20785.0 240.558095
                 Anorí    05040        46    19812.0 232.182516
               Briceño    05107        18     8599.0 209.326666
               Betulia    05093        35    16722.0 209.305107
               Argelia    76054        11     5558.0 197.912918
                 Yondó    05893        41    20870.0 196.454241
            Villa Rica    19845        42    22360.0 187.835420
             Mapiripán    50325        14     7678.0 182.339151
         Puerto Tejada    19573        81    45047.0 179.812196
             El Águila    76243        16     9185.0 174.197060
               Betania    05091        19    10985.0 172.9

## ***5. Mapas coropléticos***

Cada mapa presenta la tasa de homicidios por cada 100.000 habitantes a nivel municipal, lo que permite comparar de manera más precisa territorios con tamaños poblacionales diferentes. Además, la escala de color se encuentra limitada al percentil 95 con el fin de evitar que valores atípicos o extremos distorsionen la visualización y dificulten la interpretación de las diferencias entre la mayoría de municipios.

In [17]:
for delito in ["Homicidio", "Hurto a personas", "Delitos sexuales"]:
    datos = agrupado[(agrupado["anio"] == ultimo_anio) & (agrupado["tipo_delito"] == delito)]
    mapa = gdf_muni.merge(datos, left_on="MPIO_CCNCT", right_on="cod_muni", how="left")
    mapa["tasa"] = mapa["tasa"].fillna(0)
    tope = mapa["tasa"].quantile(0.95)

    fig = px.choropleth(
        mapa,
        geojson=mapa.geometry,
        locations=mapa.index,
        color="tasa",
        color_continuous_scale="YlOrRd",
        range_color=[0, tope],
        title=f"{delito} por cada 100,000 hab. - Colombia {ultimo_anio}",
        labels={"tasa": "Tasa x 100K"},
        hover_name="MPIO_CNMBR",
        hover_data={"DPTO_CNMBR": True, "tasa": ":.1f"},
    )

    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(width=800, height=900, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()

## ***6. Tendencia nacional***

Evolución de las tasas promedio nacionales por tipo de delito desde 2020.

In [18]:
tendencia = (
    agrupado.groupby(["anio", "tipo_delito"])["tasa"]
    .mean()
    .reset_index()
)

fig = px.line(
    tendencia,
    x="anio",
    y="tasa",
    color="tipo_delito",
    markers=True,
    title="Tendencia de tasas de criminalidad - Promedio municipal",
    labels={"tasa": "Tasa promedio x 100K", "anio": "Año", "tipo_delito": "Delito"},
)

fig.update_layout(width=900, height=500)
fig.show()

## ***7. Notas metodológicas***

- **Fuente:** Sistema de Información Estadístico, Delincuencial, Contravencional y Operativo (SIEDCO) de la Policía Nacional.
- **Población:** Proyecciones municipales del DANE basadas en el Censo Nacional de 2018 y ajustadas posteriormente a la pandemia de COVID-19.
- **Entidades territoriales:** El DANE reporta 1,122 entidades territoriales, que incluyen los 1,103 municipios oficiales y corregimientos departamentales ubicados en zonas no municipalizadas.
- **Tasa per cápita:** Calculada mediante la fórmula:

$$\text{Tasa} = \frac{\text{Número de delitos}}{\text{Población municipal}} \times 100{,}000$$

- **Limitaciones:**
  - Los datos representan delitos reportados y registrados oficialmente, por lo que no reflejan la totalidad de la criminalidad existente (cifra oscura).
  - Los municipios con poblaciones reducidas pueden presentar tasas altamente volátiles ante pequeñas variaciones en el número de casos.
  - La información correspondiente a 2025 y 2026 puede encontrarse incompleta o sujeta a actualización al momento de la consulta.